In [1]:
import pandas as pd
import numpy as np

In [2]:
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")
links = pd.read_csv("links.csv")
tags = pd.read_csv("tags.csv")

In [3]:
ratings.head()
movies.head()
links.head()
tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [4]:
ratings.info()
movies.info()
links.info()
tags.info()

<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB
<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 228.5 KB
<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9742 non-null   int64  
 1   imdbId   9742 non-null   int64  
 2   tmdbId   9734 non-null   float

In [5]:
ratings.isnull().sum()
movies.isnull().sum()
links.isnull().sum()

movieId    0
imdbId     0
tmdbId     8
dtype: int64

In [6]:
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)')
movies['year'] = pd.to_numeric(movies['year'], errors='coerce')

In [7]:
movies['title'] = movies['title'].str.replace(r'\(\d{4}\)', '', regex=True).str.strip()

In [8]:
df = pd.merge(ratings, movies, on="movieId", how="left")

In [9]:
df = pd.merge(df, links, on="movieId", how="left")

In [10]:
df.head()

,userId,movieId,rating,timestamp,title,genres,year,imdbId,tmdbId
0,1,1,4.0,964982703,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995.0,114709,862.0
1,1,3,4.0,964981247,Grumpier Old Men,Comedy|Romance,1995.0,113228,15602.0
2,1,6,4.0,964982224,Heat,Action|Crime|Thriller,1995.0,113277,949.0
3,1,47,5.0,964983815,Seven (a.k.a. Se7en),Mystery|Thriller,1995.0,114369,807.0
4,1,50,5.0,964982931,"Usual Suspects, The",Crime|Mystery|Thriller,1995.0,114814,629.0


In [11]:
df = df[['userId','movieId','rating','timestamp','title','genres','imdbId','tmdbId','year']]

In [12]:
df = df.dropna(subset=['title','rating'])

In [13]:
df['imdbId'] = df['imdbId'].fillna(0)
df['tmdbId'] = df['tmdbId'].fillna(0)

In [14]:
df['imdbId'] = df['imdbId'].astype(int)
df['tmdbId'] = df['tmdbId'].astype(int)

In [15]:
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')

In [16]:
df = df.sort_values(by=['userId','movieId'])

In [17]:
df.info()
df.head()
df.shape

<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 9 columns):
 #   Column     Non-Null Count   Dtype        
---  ------     --------------   -----        
 0   userId     100836 non-null  int64        
 1   movieId    100836 non-null  int64        
 2   rating     100836 non-null  float64      
 3   timestamp  100836 non-null  datetime64[s]
 4   title      100836 non-null  str          
 5   genres     100836 non-null  str          
 6   imdbId     100836 non-null  int64        
 7   tmdbId     100836 non-null  int64        
 8   year       100818 non-null  float64      
dtypes: datetime64[s](1), float64(2), int64(4), str(2)
memory usage: 6.9 MB


(100836, 9)

In [18]:
df.to_csv("analysis.csv", index=False)

In [19]:
# Remove trailing ', The' from titles
df['title'] = df['title'].str.replace(r', The$', '', regex=True)

In [20]:
df['title'] = df['title'].str.replace(r'^(.*), The$', r'The \1', regex=True)

In [22]:
df['title'].head(23)

0                              Toy Story
1                       Grumpier Old Men
2                                   Heat
3                   Seven (a.k.a. Se7en)
4                         Usual Suspects
5                    From Dusk Till Dawn
6                          Bottle Rocket
7                             Braveheart
8                                Rob Roy
9                         Canadian Bacon
10                             Desperado
11                         Billy Madison
12                                Clerks
13       Dumb & Dumber (Dumb and Dumber)
14                               Ed Wood
15    Star Wars: Episode IV - A New Hope
16                          Pulp Fiction
17                              Stargate
18                             Tommy Boy
19              Clear and Present Danger
20                          Forrest Gump
21                           Jungle Book
22                                  Mask
Name: title, dtype: str

In [23]:
# Save the cleaned and updated dataset
df.to_csv("analysis.csv", index=False)